**題目1.：**至ITC-Trade Map網站下載近十年韓國對美國及中國Product code四碼、六碼的商品出口、進口數據，並整理成Excel檔，自行判斷是否區分工作表儲存資料

Ø   網址https://www.trademap.org/Index.aspx

Ø   說明：以Product code六碼/進口為例，該網站選單設定如下(先選擇Country與Partner後才會進入該頁面，請自行註冊帳號登入)

**我測試Login太頻繁所以被ban，但login 成功後無法取資料。之後可以透過不同user agent proxy等方式加強反爬蟲**

In [2]:
!pip install selenium webdriver-manager
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import time

def init_driver():
    options = webdriver.ChromeOptions()
    # options.add_argument("--headless") # 若需要無頭模式，取消註解此行
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    return driver

def login_to_trade_map(driver, username, password):
    
    driver.get("https://www.trademap.org/Index.aspx")
    
    time.sleep(2) 

    # 等待並點擊登錄按鈕
    WebDriverWait(driver, 10).until(
        EC.element_to_be_clickable((By.ID, "ctl00_MenuControl_Label_Login"))
    ).click()

    # 等待 Username 和 Password 欄位
    WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.ID, "Username"))
    ).send_keys(username)

    WebDriverWait(driver, 5).until(
        EC.presence_of_element_located((By.ID, "Password"))
    ).send_keys(password)
    
    time.sleep(2) 

    # 點擊登錄按鈕
    WebDriverWait(driver, 5).until(
        EC.element_to_be_clickable((By.XPATH, "//button[contains(text(), 'Login')]"))
    ).click()

    # 檢查是否成功登錄
    WebDriverWait(driver, 5).until(EC.title_contains("Trade Map"))
    if "Trade Map" in driver.title:
        print("Login successful!")
    else:
        print("Login failed.")

def main():
    username = "ponggung1986@gmail.com" # 替換為您的帳號
    password = "W8Ce5wW5z2qMs2K" # 替換為您的密碼
 

    driver = init_driver()
    login_to_trade_map(driver, username, password)

if __name__ == "__main__":
    main()

Login successful!


**題目2.：**至台灣水利署網站下載過去所有水庫的有效蓄水量(萬立方公尺)、蓄水量百分比(%)的數據，並以日期排列整理成Excel檔，自行判斷是否區分工作表儲存數據

Ø   網址https://fhy.wra.gov.tw/fhyv2/Monitor/Reservoir

我只抓30天但覺得太慢，之後會參考 多執行續、不同proxy 等快速爬蟲工具

In [ ]:
!pip install selenium beautifulsoup4 pandas openpyxl
import random
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select, WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from bs4 import BeautifulSoup
import time
from datetime import datetime, timedelta

# 隨機 User-Agent 列表
USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/113.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/112.0.0.0 Safari/537.36",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/111.0.0.0 Safari/537.36",
]

# 隨機選擇一個 User-Agent
def get_random_user_agent():
    return random.choice(USER_AGENTS)

# 配置 WebDriver
def setup_webdriver():
    options = webdriver.ChromeOptions()
    options.add_argument("--headless")  # 無頭模式
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument(f"user-agent={get_random_user_agent()}")
    service = Service(r"C:\Users\anaco\chromedriver-win64\chromedriver-win64\chromedriver.exe")  # 替換為你的 ChromeDriver 路徑
    driver = webdriver.Chrome(service=service, options=options)
    driver.set_window_size(random.randint(1200, 1920), random.randint(800, 1080))  # 隨機窗口大小
    return driver

# 抓取指定日期的數據，增加重試機制
def fetch_data_for_date(driver, target_date, retries=3):
    all_data = []
    for attempt in range(retries):
        try:
            # 設置年份、月份和日期
            year_select = Select(driver.find_element(By.ID, "ctl00_cphMain_ucDate_cboYear"))
            month_select = Select(driver.find_element(By.ID, "ctl00_cphMain_ucDate_cboMonth"))
            day_select = Select(driver.find_element(By.ID, "ctl00_cphMain_ucDate_cboDay"))

            # 選擇年份、月份和日期
            year_select.select_by_visible_text(str(target_date.year))
            month_select.select_by_visible_text(str(target_date.month))
            day_select.select_by_visible_text(str(target_date.day))

            # 點擊查詢按鈕
            query_button = driver.find_element(By.ID, "ctl00_cphMain_btnQuery")
            driver.execute_script("arguments[0].click();", query_button)  # 使用 JavaScript 點擊避免遮罩層問題

            # 顯式等待表格加載完成
            WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.ID, "ctl00_cphMain_gvList"))
            )

            # 解析頁面內容
            html = driver.page_source
            soup = BeautifulSoup(html, "html.parser")

            # 定位目標表格
            table = soup.find("table", {"id": "ctl00_cphMain_gvList"})

            # 提取表格數據
            rows = table.find_all("tr")[2:]  # 跳過表頭
            for row in rows:
                cols = row.find_all("td")
                cols = [col.text.strip() for col in cols]
                if len(cols) >= 4:
                    reservoir_name = cols[0]  # 水庫名稱
                    effective_storage = cols[-2]  # 有效蓄水量
                    percentage = cols[-1]  # 蓄水量百分比
                    all_data.append([target_date.strftime('%Y-%m-%d'), reservoir_name, effective_storage, percentage])
            # 如果成功抓取則跳出重試
            return all_data

        except Exception as e:
            print(f"抓取 {target_date.strftime('%Y-%m-%d')} 數據失敗，嘗試次數 {attempt + 1}/{retries}")
            if attempt < retries - 1:
                # 隨機延遲後重試
                time.sleep(random.uniform(1, 3))
            else:
                print(f"抓取 {target_date.strftime('%Y-%m-%d')} 數據最終失敗。")

    return all_data

# 保存數據到 Excel 文件
def save_to_excel(data, output_file):
    df = pd.DataFrame(data, columns=["日期", "水庫名稱", "有效蓄水量(萬立方公尺)", "蓄水量百分比(%)"])
 
    df["日期"] = pd.to_datetime(df["日期"])
    # 按 "日期" 降序排序
    df = df.sort_values(by="日期", ascending=False)
    
    grouped = df.groupby("水庫名稱")
    with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
        for name, group in grouped:
            group.to_excel(writer, sheet_name=name, index=False)
    print(f"數據已按水庫名稱存入文件：{output_file}")

# 主程式
def main():
    start_time = time.time()  # 紀錄開始時間
    driver = setup_webdriver()
    try:
        # 前往目標網頁
        url = "https://fhy.wra.gov.tw/ReservoirPage_2011/StorageCapacity.aspx"
        driver.get(url)
        time.sleep(3)  # 等待頁面加載完成

        # 設置數據存儲和時間範圍
        all_data = []
        today = datetime.now()
        start_date = today - timedelta(days=1 * 30)  # 過去 30 天

        # 迴圈抓取每日數據
        current_date = start_date
        while current_date <= today:
            print(f"抓取 {current_date.strftime('%Y-%m-%d')} 的數據...")
            daily_data = fetch_data_for_date(driver, current_date)
            all_data.extend(daily_data)

            # 隨機延遲避免反爬
            delay = random.uniform(1, 2)
            time.sleep(delay)

            # 前進到下一天
            current_date += timedelta(days=1)

        # 保存為 Excel 文件
        save_to_excel(all_data, "水庫數據分組.xlsx")

    finally:
        driver.quit()

    # 紀錄結束時間並計算總時間
    end_time = time.time()
    elapsed_time = end_time - start_time
    print(f"程式執行總時間：{elapsed_time:.2f} 秒")

# 執行主程式
if __name__ == "__main__":
    main()